# Part 2 — Features and validation

So Part1 gave us a clean monthly table with **real** PM2.5 only. Here we turn that into something sklearn can eat: **lags**, **season signals**, and two kinds of **train/test split** (time + space).

**Why two splits?** A random split lets the model cheat (same station in train and test). **Time split** = “can we predict future months?” **Spatial split** = “can we predict stations the model never saw?”

**Input:** `datasets/part1_openaq_base.csv` from `part1_clean_monthly_no_annual_fill.ipynb`.

**Output:** `datasets/part2_features.csv` for Part 3.


### Imports

pandas, numpy, and `GroupShuffleSplit` (keeps whole stations together in the spatial test)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit

pd.set_option("display.max_columns", 200)
print("Libraries loaded")


Libraries loaded


### Read the Part 1 CSV

Handoff from Part 1. You should see `openaq_id`, `date`, `PM2_5`, plus weather/context. If the row count is tiny, rerun Part 1 first.


In [2]:
ROOT = Path(".")
INPUT = ROOT / "datasets" / "part1_openaq_base.csv"
OUTPUT = ROOT / "datasets" / "part2_features.csv"

df = pd.read_csv(INPUT, parse_dates=["date"])
print("Loaded:", INPUT)
print("Shape:", df.shape)
display(df.head())


Loaded: datasets/part1_openaq_base.csv
Shape: (3986, 21)


,openaq_id,eoi_code,Year,Month,date,Season,PM2_5,PM10,NO2,O3,Temp_Mean,Wind_Speed,Precipitation,Altitude,Latitude,Longitude,Station_Type,Station_Area,City,Green_Ratio,Population_Density
0,IT1975A,IT1975A,2020,2,2020-02-01,Winter,25.70,29.3,27.8,47.3,7.737931,8.710345,6.5,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0
1,IT1975A,IT1975A,2020,3,2020-03-01,Spring,15.30,19.7,NaN,71.5,8.509677,8.977419,90.6,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0
2,IT1975A,IT1975A,2020,4,2020-04-01,Spring,8.17,12.3,NaN,60.8,14.156667,7.590000,22.6,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0
3,IT2250A,IT2250A,2020,1,2020-01-01,Winter,8.50,17.9,13.7,NaN,9.087097,9.038710,38.4,3.0,42.1022,11.7844,Traffic,Urban,Civitavecchia,0.811256,699.0
4,IT2250A,IT2250A,2020,2,2020-02-01,Winter,9.68,19.5,11.2,NaN,10.903448,11.848276,12.2,3.0,42.1022,11.7844,Traffic,Urban,Civitavecchia,0.811256,699.0


### Sort by station, then time

**Must-do before lags:** `shift()` only works if each station’s rows are in chronological order. We sort by `openaq_id` and `date`, then `reset_index(drop=True)`.


In [3]:
df = df.sort_values(["openaq_id", "date"]).reset_index(drop=True)
display(df[["openaq_id", "date", "PM2_5"]].head(10))


,openaq_id,date,PM2_5
0,IT0459A,2020-04-01,8.32
1,IT0459A,2020-05-01,6.41
2,IT0459A,2020-06-01,8.42
3,IT0459A,2020-07-01,8.13
4,IT0459A,2020-08-01,7.74
5,IT0459A,2020-09-01,7.20
6,IT0459A,2020-10-01,24.90
7,IT0459A,2020-11-01,18.40
8,IT0459A,2020-12-01,21.50
9,IT0459A,2021-01-01,22.10


### Calendar / season features

Month index plus **sin/cos** so December is “near” January in a smooth way. Trees can learn season without this, but it helps Ridge and costs almost nothing.


In [4]:
df["month_num"] = df["Month"]
df["month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)
display(df[["Month", "month_num", "month_sin", "month_cos"]].head())


,Month,month_num,month_sin,month_cos
0,4,4,8.660254e-01,-0.500000
1,5,5,5.000000e-01,-0.866025
2,6,6,1.224647e-16,-1.000000
3,7,7,-5.000000e-01,-0.866025
4,8,8,-8.660254e-01,-0.500000


### PM2.5 lags and short rolling stats

We use **past** months only (`shift(1)`, rolling on shifted series) so we don’t leak the current month’s PM2.5 into the features.

Lags are usually strong here — that’s normal for monthly air data.


In [5]:
group = df.groupby("openaq_id", group_keys=False)
df["PM2_5_lag1"] = group["PM2_5"].shift(1)
df["PM2_5_lag2"] = group["PM2_5"].shift(2)
df["PM2_5_lag3"] = group["PM2_5"].shift(3)
df["PM2_5_roll3_mean"] = group["PM2_5"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
df["PM2_5_roll3_std"] = group["PM2_5"].shift(1).rolling(3).std().reset_index(level=0, drop=True)
display(df[["openaq_id", "date", "PM2_5", "PM2_5_lag1", "PM2_5_lag2", "PM2_5_lag3", "PM2_5_roll3_mean"]].head(12))


,openaq_id,date,PM2_5,PM2_5_lag1,PM2_5_lag2,PM2_5_lag3,PM2_5_roll3_mean
0,IT0459A,2020-04-01,8.32,NaN,NaN,NaN,NaN
1,IT0459A,2020-05-01,6.41,8.32,NaN,NaN,NaN
2,IT0459A,2020-06-01,8.42,6.41,8.32,NaN,NaN
3,IT0459A,2020-07-01,8.13,8.42,6.41,8.32,7.716667
4,IT0459A,2020-08-01,7.74,8.13,8.42,6.41,7.653333
5,IT0459A,2020-09-01,7.20,7.74,8.13,8.42,8.096667
6,IT0459A,2020-10-01,24.90,7.20,7.74,8.13,7.690000
7,IT0459A,2020-11-01,18.40,24.90,7.20,7.74,13.280000
8,IT0459A,2020-12-01,21.50,18.40,24.90,7.20,16.833333
9,IT0459A,2021-01-01,22.10,21.50,18.40,24.90,21.600000


### Weather lag (previous month)

`Temp_Mean_lag1`, etc. tie this month’s target to **last month’s** weather. Missing values can happen; Part3’s pipeline imputes.


In [6]:
for c in ["Temp_Mean", "Wind_Speed", "Precipitation"]:
    if c in df.columns:
        df[f"{c}_lag1"] = group[c].shift(1)

weather_lag_cols = [c for c in df.columns if c.endswith("_lag1") and ("Temp" in c or "Wind" in c or "Precipitation" in c)]
print("Weather lag columns:", weather_lag_cols)
display(df[["openaq_id", "date"] + weather_lag_cols].head(10))


Weather lag columns: ['Temp_Mean_lag1', 'Wind_Speed_lag1', 'Precipitation_lag1']


,openaq_id,date,Temp_Mean_lag1,Wind_Speed_lag1,Precipitation_lag1
0,IT0459A,2020-04-01,NaN,NaN,NaN
1,IT0459A,2020-05-01,13.490000,10.310000,44.6
2,IT0459A,2020-06-01,18.061290,10.125806,62.0
3,IT0459A,2020-07-01,21.836667,9.896667,59.2
4,IT0459A,2020-08-01,24.532258,9.580645,28.6
5,IT0459A,2020-09-01,25.606452,10.080645,65.2
6,IT0459A,2020-10-01,20.910000,9.833333,57.8
7,IT0459A,2020-11-01,14.580645,8.735484,52.8
8,IT0459A,2020-12-01,11.090000,9.396667,48.7
9,IT0459A,2021-01-01,7.819355,10.661290,82.2


### Other pollutants lag1 

PM10 / NO2 / O3 from the **previous** month as extra signal. Lots of NaNs is ok for predictors; the target must stay clean.


In [7]:
for c in ["PM10", "NO2", "O3"]:
    if c in df.columns:
        df[f"{c}_lag1"] = group[c].shift(1)

poll_lag_cols = [c for c in ["PM10_lag1", "NO2_lag1", "O3_lag1"] if c in df.columns]
print("Pollutant lag columns:", poll_lag_cols)
display(df[["openaq_id", "date"] + poll_lag_cols].head(10))


Pollutant lag columns: ['PM10_lag1', 'NO2_lag1', 'O3_lag1']


,openaq_id,date,PM10_lag1,NO2_lag1,O3_lag1
0,IT0459A,2020-04-01,NaN,NaN,NaN
1,IT0459A,2020-05-01,21.4,24.4,54.1
2,IT0459A,2020-06-01,19.7,25.1,56.7
3,IT0459A,2020-07-01,21.6,23.9,61.4
4,IT0459A,2020-08-01,20.4,20.1,56.5
5,IT0459A,2020-09-01,20.0,27.2,53.0
6,IT0459A,2020-10-01,16.9,28.6,35.6
7,IT0459A,2020-11-01,36.4,29.3,26.0
8,IT0459A,2020-12-01,26.5,26.3,29.1
9,IT0459A,2021-01-01,26.7,19.9,32.5


### Drop rows without enough history

We throw away the first months of each station’s record because there isn’t enough past data yet to compute “last month /2 months ago / 3 months ago” PM2.5; that’s why you see fewer rows than in Part 1.


In [8]:
required_lags = ["PM2_5_lag1", "PM2_5_lag2", "PM2_5_lag3"]
df_model = df.dropna(subset=required_lags).copy()
print("Rows before:", len(df))
print("Rows after lag filter:", len(df_model))
display(df_model.head())


Rows before: 3986
Rows after lag filter: 3554


,openaq_id,eoi_code,Year,Month,date,Season,PM2_5,PM10,NO2,O3,Temp_Mean,Wind_Speed,Precipitation,Altitude,Latitude,Longitude,Station_Type,Station_Area,City,Green_Ratio,Population_Density,month_num,month_sin,month_cos,PM2_5_lag1,PM2_5_lag2,PM2_5_lag3,PM2_5_roll3_mean,PM2_5_roll3_std,Temp_Mean_lag1,Wind_Speed_lag1,Precipitation_lag1,PM10_lag1,NO2_lag1,O3_lag1
3,IT0459A,IT0459A,2020,7,2020-07-01,Summer,8.13,20.4,20.1,56.5,24.532258,9.580645,28.6,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,7,-0.500000,-8.660254e-01,8.42,6.41,8.32,7.716667,1.132711,21.836667,9.896667,59.2,21.6,23.9,61.4
4,IT0459A,IT0459A,2020,8,2020-08-01,Summer,7.74,20.0,27.2,53.0,25.606452,10.080645,65.2,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,8,-0.866025,-5.000000e-01,8.13,8.42,6.41,7.653333,1.086477,24.532258,9.580645,28.6,20.4,20.1,56.5
5,IT0459A,IT0459A,2020,9,2020-09-01,Autumn,7.20,16.9,28.6,35.6,20.910000,9.833333,57.8,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,9,-1.000000,-1.836970e-16,7.74,8.13,8.42,8.096667,0.341223,25.606452,10.080645,65.2,20.0,27.2,53.0
6,IT0459A,IT0459A,2020,10,2020-10-01,Autumn,24.90,36.4,29.3,26.0,14.580645,8.735484,52.8,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,10,-0.866025,5.000000e-01,7.20,7.74,8.13,7.690000,0.467012,20.910000,9.833333,57.8,16.9,28.6,35.6
7,IT0459A,IT0459A,2020,11,2020-11-01,Autumn,18.40,26.5,26.3,29.1,11.090000,9.396667,48.7,15.0,43.5989,13.3419,Background,Suburban,Chiaravalle,0.813059,788.0,11,-0.500000,8.660254e-01,24.90,7.20,7.74,13.280000,10.066837,14.580645,8.735484,52.8,36.4,29.3,26.0


### Missing values after features

Quick sanity check before splits. If half the table is NaN, we probably broke a column name or merge.


In [9]:
missing_pct = (df_model.isna().mean() * 100).sort_values(ascending=False).round(2)
print("Missing % after feature creation:")
print(missing_pct.head(25))


Missing % after feature creation:
O3                    47.19
O3_lag1               47.13
PM10                  15.67
PM10_lag1             15.62
Population_Density     2.28
NO2                    2.25
NO2_lag1               2.25
Temp_Mean_lag1         0.00
Wind_Speed_lag1        0.00
Green_Ratio            0.00
Precipitation_lag1     0.00
PM2_5_roll3_mean       0.00
PM2_5_lag3             0.00
PM2_5_lag2             0.00
PM2_5_lag1             0.00
month_cos              0.00
month_sin              0.00
month_num              0.00
PM2_5_roll3_std        0.00
openaq_id              0.00
City                   0.00
eoi_code               0.00
Station_Type           0.00
Longitude              0.00
Latitude               0.00
dtype: float64


### Time split (train on the past, test on the future)

Cutoff = **80th percentile** of `date` (roughly last ~20% of timeline in test). 


In [10]:
cutoff_date = df_model["date"].quantile(0.8)
train_time = df_model[df_model["date"] <= cutoff_date].copy()
test_time = df_model[df_model["date"] > cutoff_date].copy()

print("Cutoff date:", cutoff_date.date())
print("Train rows:", len(train_time))
print("Test rows:", len(test_time))
print("Train date range:", train_time["date"].min().date(), "to", train_time["date"].max().date())
print("Test date range:", test_time["date"].min().date(), "to", test_time["date"].max().date())


Cutoff date: 2025-04-01
Train rows: 2885
Test rows: 669
Train date range: 2020-07-01 to 2025-04-01
Test date range: 2025-05-01 to 2025-11-01


### Spatial split (hide entire stations)

`GroupShuffleSplit` puts every row of a station on **one** side only. Normal random split might do this:

Train: IT0459A in Jan–Jun, IT1975A in Mar–Aug, …
Test: IT0459A in Jul–Dec, …

So the same station appears in both train and test. The model can learn things like “this station is usually a bit high” or “this station’s winter pattern,” and that helps test accuracy even for “new” months.

GroupShuffleSplit with groups=openaq_id picks whole stations and assigns each station entirely to train or entirely to test. So if IT0459A is in test, no row for IT0459A appears in train.

So every row of a station is on one side only = the test set is stations the model has never seen during training.


In [11]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_train, idx_test = next(splitter.split(df_model, groups=df_model["openaq_id"]))
train_space = df_model.iloc[idx_train].copy()
test_space = df_model.iloc[idx_test].copy()

print("Spatial train rows:", len(train_space))
print("Spatial test rows:", len(test_space))
print("Train stations:", train_space["openaq_id"].nunique())
print("Test stations:", test_space["openaq_id"].nunique())


Spatial train rows: 2831
Spatial test rows: 723
Train stations: 91
Test stations: 23


### Feature list for Part 3

Only columns that exist get passed through. Ridge + Random Forest in Part 3 use this list + target `PM2_5`.


In [12]:
candidate_features = [
    "PM2_5_lag1", "PM2_5_lag2", "PM2_5_lag3", "PM2_5_roll3_mean", "PM2_5_roll3_std",
    "PM10_lag1", "NO2_lag1", "O3_lag1",
    "Temp_Mean_lag1", "Wind_Speed_lag1", "Precipitation_lag1",
    "month_num", "month_sin", "month_cos",
    "Green_Ratio", "Latitude", "Longitude"
]

feature_cols = [c for c in candidate_features if c in df_model.columns]
print("Feature count:", len(feature_cols))
print(feature_cols)


Feature count: 17
['PM2_5_lag1', 'PM2_5_lag2', 'PM2_5_lag3', 'PM2_5_roll3_mean', 'PM2_5_roll3_std', 'PM10_lag1', 'NO2_lag1', 'O3_lag1', 'Temp_Mean_lag1', 'Wind_Speed_lag1', 'Precipitation_lag1', 'month_num', 'month_sin', 'month_cos', 'Green_Ratio', 'Latitude', 'Longitude']


### Save `part2_features.csv`

Regenerate this file after changing Part 1 — avoid hand-editing in Excel (decimals/encoding).


In [13]:
save_cols = ["openaq_id", "date", "Year", "Month", "PM2_5"] + feature_cols
save_cols = [c for c in save_cols if c in df_model.columns]

df_model[save_cols].to_csv(OUTPUT, index=False)
print("Saved:", OUTPUT)
print("Final shape:", df_model[save_cols].shape)
display(df_model[save_cols].head())


Saved: datasets/part2_features.csv
Final shape: (3554, 22)


,openaq_id,date,Year,Month,PM2_5,PM2_5_lag1,PM2_5_lag2,PM2_5_lag3,PM2_5_roll3_mean,PM2_5_roll3_std,PM10_lag1,NO2_lag1,O3_lag1,Temp_Mean_lag1,Wind_Speed_lag1,Precipitation_lag1,month_num,month_sin,month_cos,Green_Ratio,Latitude,Longitude
3,IT0459A,2020-07-01,2020,7,8.13,8.42,6.41,8.32,7.716667,1.132711,21.6,23.9,61.4,21.836667,9.896667,59.2,7,-0.500000,-8.660254e-01,0.813059,43.5989,13.3419
4,IT0459A,2020-08-01,2020,8,7.74,8.13,8.42,6.41,7.653333,1.086477,20.4,20.1,56.5,24.532258,9.580645,28.6,8,-0.866025,-5.000000e-01,0.813059,43.5989,13.3419
5,IT0459A,2020-09-01,2020,9,7.20,7.74,8.13,8.42,8.096667,0.341223,20.0,27.2,53.0,25.606452,10.080645,65.2,9,-1.000000,-1.836970e-16,0.813059,43.5989,13.3419
6,IT0459A,2020-10-01,2020,10,24.90,7.20,7.74,8.13,7.690000,0.467012,16.9,28.6,35.6,20.910000,9.833333,57.8,10,-0.866025,5.000000e-01,0.813059,43.5989,13.3419
7,IT0459A,2020-11-01,2020,11,18.40,24.90,7.20,7.74,13.280000,10.066837,36.4,29.3,26.0,14.580645,8.735484,52.8,11,-0.500000,8.660254e-01,0.813059,43.5989,13.3419


### Next: Part 3

Open **`part3_modeling_and_shap.ipynb`**: train Ridge & Random Forest, plots, SHAP, and a printed **champion** model.
